# Weber BLL — Cumulative Regret Evaluation

## What is Weber BLL?

**Weber et al. (2018), "Optimizing over a Bayesian Last Layer"** proposes a method for training neural network feature extractors in a bandit setting. The architecture has two components:

1. **Feature extractor** $f_\theta(x)$: maps context $x$ to features $h$ (linear or MLP)
2. **Bayesian linear last layer**: per-arm regression $z_n \sim \mathcal{N}(\mu_n, \Sigma_n)$ on top of features, with closed-form posterior updates

The key idea: when retraining the feature extractor, instead of optimizing the log-likelihood at the posterior **mean** $\hat{\theta}$, Weber marginalizes over **all** values of the last layer, weighted by the posterior.

### Closed-form marginalized gradient (Gaussian case)

For arm $n$ with posterior $z_n \sim \mathcal{N}(\mu_n, \Sigma_n)$, features $h$, and target $y_n$:

$$\nabla_h \mathbb{E}_{z_n}[\log p(y_n \mid z_n, h)] = -\frac{1}{\sigma^2}\left[(\mu_n^\top h - y_n)\mu_n + \Sigma_n h\right]$$

Two terms:
- **MSE gradient**: $(\text{pred}_n - y_n) \cdot \mu_n$ — standard prediction error signal
- **Uncertainty correction**: $\Sigma_n h$ — the Weber contribution! Pushes features toward directions where the posterior is uncertain

### Algorithm structure (strictly phased)

- **Online phase** (features frozen): Update Bayesian posterior, use UCB for exploration
- **Offline phase** (every `phase_length` rounds): Retrain features with marginalized LL + diagonal Fisher regularization, then replay all buffered data to rebuild posteriors

### How it differs from BayesianSPO

| | BayesianSPO | Weber BLL |
|---|---|---|
| **Loss** | SPO+ (decision-focused) | Gaussian log-likelihood |
| **Feature updates** | Every round | Phased (batch retraining) |
| **Regularization** | None | Diagonal Fisher (prevents catastrophic forgetting) |
| **Marginalization** | Point estimate (posterior mean) | Closed-form over full posterior |

## Experiment setup

Settings matched exactly to the existing toy notebooks (`KS_DI_Improvement.ipynb` and `SP_DI_Improvement.ipynb`):

| | Knapsack | Shortest Path |
|---|---|---|
| **Env** | `KnapsackEnv(n_items=32, context_dim=5, capacity=20, noise_width=0.25)` | `ShortestPathEdgeEnv(grid_size=8, context_dim=5, noise_width=0.25)` |
| **lr** | 0.01 | 0.001 |
| **hidden_dim** | 32 | 32 |
| **lambda_reg** | 1.0 | 1.0 |
| **beta** | 1.0 (negated for minimize) | 1.0 (negated → LCB) |
| **n_rounds** | 1000 | 2000 |
| **seeds** | [0, 1, 2, 5, 6] | [0, 1, 2, 5, 6] |
| **feedback** | semi-bandit | semi-bandit |

Agents compared:
- **WeberBLL**: Faithful Weber implementation (log-likelihood, phased, Fisher reg)
- **BayesianSPO**: Decision-focused features (SPO+ loss, per-round updates, mask_grad=True)
- **BayesianMSE**: Prediction-focused features (MSE loss, per-round updates)
- **LinUCB**: Linear UCB baseline (no feature learning)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import time

from src.envs.knapsack import KnapsackEnv
from src.envs.shortest_path import ShortestPathEdgeEnv
from src.agents.shared_context import (
    BayesianSPOAgent, BayesianMSEAgent, LinUCBAgent,
)
from src.agents.shared_context.weber_bll import WeberBLLAgent
from src.agents.base import run_experiment

In [ ]:
# ══════════════════════════════════════════════════════════════
# Settings — matched to KS_DI_Improvement.ipynb & SP_DI_Improvement.ipynb
# ══════════════════════════════════════════════════════════════

DATASETS = {
    'knapsack': dict(
        env_cls=KnapsackEnv,
        env_kwargs=dict(n_items=32, context_dim=5, capacity=20.0, noise_width=0.25),
        minimize=False,
        # HPs from KS_DI_Improvement.ipynb
        hp=dict(lr=0.01, hidden_dim=32, lambda_reg=1.0, beta=1.0),
        n_rounds=1000,
    ),
    'shortest_path': dict(
        env_cls=ShortestPathEdgeEnv,
        env_kwargs=dict(grid_size=8, context_dim=5, noise_width=0.25),
        minimize=True,
        # HPs from SP_DI_Improvement.ipynb
        hp=dict(lr=0.001, hidden_dim=32, lambda_reg=1.0, beta=1.0),
        n_rounds=2000,
    ),
}

# Same seeds as the toy notebooks
SEEDS = [0, 1, 2, 5, 6]

# ══════════════════════════════════════════════════════════════
# Run
# ══════════════════════════════════════════════════════════════

all_trajectories = {}

for ds_name, ds_cfg in DATASETS.items():
    env_cls = ds_cfg['env_cls']
    env_kwargs = ds_cfg['env_kwargs']
    minimize = ds_cfg['minimize']
    hp = ds_cfg['hp']
    n_rounds = ds_cfg['n_rounds']
    
    bayes_beta = -hp['beta'] if minimize else hp['beta']
    
    print(f'\n{"="*60}')
    print(f'{ds_name.upper()}')
    print(f'  env: {env_cls.__name__}({env_kwargs})')
    print(f'  hp:  lr={hp["lr"]}, hidden_dim={hp["hidden_dim"]}, '
          f'lambda_reg={hp["lambda_reg"]}, beta={hp["beta"]}')
    print(f'  n_rounds={n_rounds}, seeds={SEEDS}, minimize={minimize}')
    print(f'{"="*60}')
    
    for seed in SEEDS:
        env = env_cls(**env_kwargs, seed=seed)
        solver = env.solve
        
        common = dict(
            context_dim=env.context_dim, n_arms=env.n_arms,
            weights=env.weights, capacity=env.capacity,
        )
        
        agents = {
            'LinUCB': LinUCBAgent(
                **common, alpha=hp['beta'], lambda_reg=hp['lambda_reg'],
                minimize=minimize, solver=solver,
            ),
            'BayesianMSE': BayesianMSEAgent(
                **common, hidden_dim=hp['hidden_dim'], beta=bayes_beta,
                lambda_reg=hp['lambda_reg'], decay_rounds=n_rounds,
                solver=solver, seed=seed,
            ),
            'BayesianSPO': BayesianSPOAgent(
                **common, hidden_dim=hp['hidden_dim'], beta=bayes_beta,
                decay_rounds=n_rounds, lambda_reg=hp['lambda_reg'],
                minimize=minimize, solver=solver, seed=seed,
                mask_grad=True,
            ),
            'WeberBLL': WeberBLLAgent(
                **common, hidden_dim=hp['hidden_dim'], beta=bayes_beta,
                decay_rounds=n_rounds, lambda_reg=hp['lambda_reg'],
                lr=hp['lr'], minimize=minimize, solver=solver, seed=seed,
                mask_grad=True,
            ),
        }
        
        print(f'\n  seed={seed}:', end='')
        for agent_name, agent in agents.items():
            t0 = time.perf_counter()
            env_run = env_cls(**env_kwargs, seed=seed)
            result = run_experiment(
                env_run, agent, n_rounds=n_rounds,
                lr=hp['lr'], full_feedback=False,
            )
            dt = time.perf_counter() - t0
            
            all_trajectories[(agent_name, ds_name, seed)] = {
                'cumulative_regret': result['cumulative_regret'],
                'regrets': result['regrets'],
            }
            print(f'  {agent_name}={result["final_regret"]:.0f}({dt:.1f}s)', end='')
        print()

print('\nDone!')

## Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

AGENT_NAMES = ['LinUCB', 'BayesianMSE', 'BayesianSPO', 'WeberBLL']
COLORS = {
    'LinUCB':      '#7f7f7f',
    'BayesianMSE': '#1f77b4',
    'BayesianSPO': '#d62728',
    'WeberBLL':    '#2ca02c',
}
LABELS = {
    'LinUCB':      'LinUCB',
    'BayesianMSE': 'BayesianMSE',
    'BayesianSPO': 'BayesianSPO',
    'WeberBLL':    'Weber BLL',
}
DS_NAMES = list(DATASETS.keys())

def get_trajectories(ds_name, agent_name, metric='cumulative_regret'):
    """Collect trajectories across seeds for one agent on one dataset."""
    trajs = []
    for s in SEEDS:
        key = (agent_name, ds_name, s)
        if key in all_trajectories:
            trajs.append(all_trajectories[key][metric])
    return np.array(trajs) if trajs else np.array([])

### Cumulative Regret Over Time

The main comparison: how total regret accumulates as agents learn. Lower is better. Shaded regions show ±1 std across seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ds_name in zip(axes, DS_NAMES):
    n_rounds = DATASETS[ds_name]['n_rounds']
    rounds = np.arange(1, n_rounds + 1)
    
    for agent_name in AGENT_NAMES:
        trajs = get_trajectories(ds_name, agent_name)
        if len(trajs) == 0:
            continue
        mean = trajs.mean(axis=0)
        std = trajs.std(axis=0)
        
        ax.plot(rounds, mean, color=COLORS[agent_name], 
                label=LABELS[agent_name], linewidth=2)
        ax.fill_between(rounds, mean - std, mean + std,
                        color=COLORS[agent_name], alpha=0.15)
    
    hp = DATASETS[ds_name]['hp']
    title = ds_name.replace('_', ' ').title()
    ax.set_title(f'{title}\n(lr={hp["lr"]}, hidden_dim={hp["hidden_dim"]}, '
                 f'n_rounds={n_rounds})', fontsize=12)
    ax.set_xlabel('Round')
    ax.set_ylabel('Cumulative Regret')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

fig.suptitle(f'Cumulative Regret (Semi-Bandit, {len(SEEDS)} seeds, toy-notebook HPs)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Instantaneous Regret (Smoothed)

Per-round regret smoothed with a rolling window. Shows how quickly each agent's per-round performance improves. Decreasing = agent is learning.

In [ ]:
WINDOW = 25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ds_name in zip(axes, DS_NAMES):
    n_rounds = DATASETS[ds_name]['n_rounds']
    
    for agent_name in AGENT_NAMES:
        trajs = get_trajectories(ds_name, agent_name, metric='regrets')
        if len(trajs) == 0:
            continue
        
        mean_regrets = trajs.mean(axis=0)
        smoothed = np.convolve(mean_regrets, np.ones(WINDOW)/WINDOW, mode='valid')
        rounds = np.arange(WINDOW, n_rounds + 1)
        
        ax.plot(rounds, smoothed, color=COLORS[agent_name],
                label=LABELS[agent_name], linewidth=1.5, alpha=0.85)
    
    title = ds_name.replace('_', ' ').title()
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel(f'Instant Regret ({WINDOW}-round avg)')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

fig.suptitle('Instantaneous Regret (Smoothed)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Final Regret Summary (Bar Chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ds_name in zip(axes, DS_NAMES):
    means, stds, colors, labels = [], [], [], []
    
    for agent_name in AGENT_NAMES:
        trajs = get_trajectories(ds_name, agent_name)
        if len(trajs) == 0:
            continue
        final = trajs[:, -1]
        means.append(np.mean(final))
        stds.append(np.std(final))
        colors.append(COLORS[agent_name])
        labels.append(LABELS[agent_name])
    
    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.85,
                  capsize=5, edgecolor='k', linewidth=0.5)
    
    best_idx = np.argmin(means)
    bars[best_idx].set_edgecolor('#2ca02c')
    bars[best_idx].set_linewidth(2.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha='right')
    ax.set_ylabel('Final Cumulative Regret')
    n_rounds = DATASETS[ds_name]['n_rounds']
    title = ds_name.replace('_', ' ').title()
    ax.set_title(f'{title} ({n_rounds} rounds)', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f'Final Regret ({len(SEEDS)} seeds, ±1 std)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Summary Table

In [ ]:
import pandas as pd

rows = []
for ds_name in DS_NAMES:
    for agent_name in AGENT_NAMES:
        trajs = get_trajectories(ds_name, agent_name)
        if len(trajs) == 0:
            continue
        final = trajs[:, -1]
        rows.append({
            'Dataset': ds_name.replace('_', ' ').title(),
            'Agent': LABELS[agent_name],
            'Mean Regret': f'{np.mean(final):.0f}',
            'Std': f'{np.std(final):.0f}',
            'Seeds': len(trajs),
        })

df = pd.DataFrame(rows)
df

### Regret Improvement vs BayesianSPO

Relative improvement of Weber BLL over BayesianSPO (the main decision-focused baseline).

In [ ]:
print("Relative improvement of Weber BLL over baselines:\n")
for ds_name in DS_NAMES:
    weber_final = get_trajectories(ds_name, 'WeberBLL')[:, -1].mean()
    
    print(f"{ds_name.replace('_', ' ').title()}:")
    for agent_name in ['LinUCB', 'BayesianMSE', 'BayesianSPO']:
        baseline_final = get_trajectories(ds_name, agent_name)[:, -1].mean()
        improvement = (baseline_final - weber_final) / baseline_final * 100
        print(f"  vs {LABELS[agent_name]:>15s}: {improvement:+.1f}%  "
              f"({baseline_final:.0f} → {weber_final:.0f})")
    print()